## Notebook: 01_ingestion

# SPA:
Carga el CSV de Telco Customer Churn desde el volumen de Unity Catalog y lo persiste sin modificar como tabla Delta en la capa Bronze.  

- **Entrada:** `/Volumes/telco_churn/bronze/raw/telco_churn.csv`
- **Salida:** `telco_churn.bronze.customers_raw`

## Decisiones

- Los datos se guardan tal y como vienen del origen, sin limpieza ni cambios de
  tipo. Bronze es el punto de recuperación del proyecto.
- Lectura con pandas y persistencia en Delta mediante Spark. El dataset es
  pequeño, así que no hay motivo para usar Spark en la manipulación.
- Modo de escritura `overwrite`, de forma que el notebook se puede reejecutar
  entero sin romperse.

## Verificación

Comprobaciones con `assert` antes de escribir:

- 7043 filas x 21 columnas
- `customerID` sin duplicados
- Columna objetivo `Churn` presente
- Sin valores nulos

# ENG:
Loads the Telco Customer Churn CSV from Unity Catalog volume and persists it as a Delta table in the Bronze layer.  

- **Input:** `/Volumes/telco_churn/bronze/raw/telco_churn.csv`
- **Output:** `telco_churn.bronze.customers_raw`

## Decisions

- The data is stored as it comes from the source, without cleaning or type changes. Bronze is the project recovery point.
- Reading with pandas and persisting with Spark. The dataset is small, so there is no reason to use Spark for manipulation.
- Write mode `overwrite`, so the notebook can be re-executed without breaking.

## Verification

Checks with `assert` before writing:

- 7043 rows x 21 columns
- `customerID` without duplicates
- Target column `Churn` present
- No nulls

In [0]:
import pandas as pd
from churn.config import RAW_CSV, BRONZE_TABLE, VOLUME


In [0]:
df = pd.read_csv(RAW_CSV)

print(df.shape)
df.head(10)

In [0]:
df.info()

## SPA:
Una vez cargados los datos, hacemos comprobaciones y afirmaciones que deben cumplirse para que el resto del pipeline tenga sentido.  
## ENG:
Once the data is loaded, we make checks and assertions that must be fulfilled for the rest of the pipeline to make sense.

In [0]:
assert df.shape == (7043, 21), "There are not 7043 rows and 21 columns"
assert df["customerID"].is_unique, "customerID is not unique"
assert "Churn" in df.columns, "Churn column not found"
assert df.isnull().sum().sum() == 0, "There are nulls in the dataset"

print("All tests passed // Todos los tests han sido realizados")

In [0]:
# SPA: Ahora creamos la tabla
# ENG: Now we create the table

spark.createDataFrame(df).write.mode("overwrite").saveAsTable(BRONZE_TABLE)

In [0]:
# Checks

sdf_bronze = spark.table(BRONZE_TABLE)

n_filas = sdf_bronze.count()
n_cols  = len(sdf_bronze.columns)

assert n_filas == 7043, f"Filas inesperadas en bronze: {n_filas}"
assert n_cols  == 21,   f"Columnas inesperadas en bronze: {n_cols}"

sdf_bronze.printSchema()

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}"))

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {BRONZE_TABLE}"))